## CAMP2Ex Radiosonde Plotting Code (Questions?  Email brodenkirch@wisc.edu)

#### Given a filepath (_radiosonde_folder_, see cell #2) that contains CAMP2Ex radiosonde ".txt" files, this code will:

1. Loop through each radiosonde file to filter/QC the data and add all filtered/QCed radiosonde data to the given date's (_file_date_, see cell #2) created `final_radiosonde_YYYYMMDD.csv` file, saved to _day_folder_. (cell #4)

2. Make _height vs. time_ radiosonde moisture and wind availability plots for the given _file_date_, saved to _day_folder_. (cell #5)

3. Make theta, theta-e, and theta-v plots for each radiosonde profile, saved to _day_folder_. (cell #6)

4. Make skew-T plots (not quite publication quality) for each radiosonde profile using MetPy, saved to _day_folder_. (cell #7)

### Hope this helps!  Feel free to edit the code to fit your needs.  The variables you will definitely want to change right away are _file_date_ (cell #2), _day_folder_ (cell #2), and _radiosonde_folder_ (cell #2).  Once you change these, everything should run properly as is.  If it doesn't, let the author know (brodenkirch@wisc.edu). You likely will want to edit the radiosondes in the _sondes_with_nowind_or_nomoisture_ list as well (top of cell #4).

In [ ]:
import os
import sys
import xarray as xr
import pandas as pd
import numpy as np
from datetime import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
#import matplotlib.colors as mplc

import sharppy
import sharppy.sharptab.profile as profile
#import sharppy.sharptab.interp as interp
import sharppy.sharptab.winds as winds
import sharppy.sharptab.utils as utils
import sharppy.sharptab.params as params
import sharppy.sharptab.thermo as thermo

import metpy.calc as mpcalc
import metpy.plots as mplots
from metpy.units import units

from PIL import Image
            

In [ ]:
file_date = '20190917'  #which date to create clean radiosonde data CSV for

day_folder = os.path.join(os.getcwd(), file_date)
radiosonde_folder = os.path.join(day_folder, 'Radiosonde_files')
radio_final_name = os.path.join(day_folder, 'final_radiosonde_' + file_date + '.csv')


In [ ]:
#Display the contents of a typical CAMP2Ex radiosonde TXT file
column_names = ['Seconds After Launch', 'Hour', 'Minute', 'Second', 'Pressure [mb]', 'Temperature [C]', 'Dewpoint [C]',
                'RH [%]', 'Uwind [m/s]', 'Vwind [m/s]', 'Wspd [m/s]', 'Wdir [deg]', 'dZ [m/s]', 'Geopotential Altitude [m]',
                'Longitude [deg]', 'Latitude [deg]', 'GPS Altitude [m]']
test = pd.read_csv(os.path.join('/Users/ben/Desktop/CAMP2Ex/Coding/20190917/Radiosonde_files', 'piston-sonde-hres_SONDE_201909170226_R0_99993.txt'),
                   sep = '\\s+', names = column_names, skiprows = 14, na_values = '-999.00')
test


In [ ]:
#loop through each radiosonde file to filter/QC the data and add to the given date's final_radiosonde CSV
sondes_with_nowind_or_nomoisture = ['']

column_names = ['Seconds After Launch', 'Hour', 'Minute', 'Second', 'Pressure [mb]', 'Temperature [C]', 'Dewpoint [C]',
                'RH [%]', 'Uwind [m/s]', 'Vwind [m/s]', 'Wspd [m/s]', 'Wdir [deg]', 'dZ [m/s]', 'Geopotential Altitude [m]',
                'Longitude [deg]', 'Latitude [deg]', 'GPS Altitude [m]']

first_file = True
for a in sorted(os.listdir(radiosonde_folder)):  #sorted() goes through the files in alphabetical order
    
    if a[-4:] != '.txt':  #grab only the radiosonde .nc files from the directory
        continue
    else:
        df = pd.read_csv(os.path.join(radiosonde_folder, a), sep = '\\s+', names = column_names, skiprows = 14, na_values = '-999.00')
    
    #make the radiosonde time the time when the radiosonde was deployed, instead of the time of the first good data point
    radio_time = str(int(df['Hour'].iloc[0])).zfill(2) + str(int(df['Minute'].iloc[0])).zfill(2) + str(int(df['Second'].iloc[0]) + 1).zfill(2)
    #radio_full_time = '-'.join([file_date[:4], file_date[4:6], file_date[6:]]) + ' ' + ':'.join([])
    radio_full_time = datetime.strptime(file_date + radio_time, "%Y%m%d%H%M%S")

    #The QC method below is used if no wind data or no moisture data (or at least very large gaps) may exist throughout a radiosonde
    #QC the data: alt. (hydrostatic or GPS) available and > 0, pressure > 0 (and not NaN), and (u/v wind not NaN) OR (temp/RH/dewpoint not NaN)
    df_use = df[((df['Geopotential Altitude [m]'] > 0) | (df['GPS Altitude [m]'] > 0)) & (df['Pressure [mb]'] > 0) & 
                (((df['Uwind [m/s]'].notnull()) & (df['Vwind [m/s]'].notnull())) | ((df['Temperature [C]'].notnull()) & (df['Dewpoint [C]'].notnull()) & (df['RH [%]'] >= 0)))].copy()
        #^^^ using .copy() to prevent chain-indexing --> https://www.dataquest.io/blog/settingwithcopywarning/
    
    # if a[-20:-6] in sondes_with_nowind_or_nomoisture:
    # ##The QC method below is used if no wind data or no moisture data (or at least very large gaps) may exist throughout a radiosonde
    #     #QC the data: alt. (geopotential or GPS) available and > 0, pressure > 0 (and not NaN), and (u/v wind not NaN) OR (temp/RH/dewpoint not NaN)
    #     df_use = df[((df['Geopotential Altitude [m]'] > 0) | (df['GPS Altitude [m]'] > 0)) & (df['Pressure [mb]'] > 0) & 
    #                 (((df['Uwind [m/s]'].notnull()) & (df['Vwind [m/s]'].notnull())) | ((df['Temperature [C]'].notnull()) & (df['Dewpoint [C]'].notnull()) & (df['RH [%]'] >= 0)))].copy()
    #         #^^^ using .copy() to prevent chain-indexing --> https://www.dataquest.io/blog/settingwithcopywarning/
    # else:
    #     #QC the data: alt. (geopotential or GPS) available and > 0, pressure > 0 (and not NaN), u/v/temp/RH/dewpoint not NaN
    #     df_use = df[((df['Geopotential Altitude [m]'] > 0) | (df['GPS Altitude [m]'] > 0)) & (df['Pressure [mb]'] > 0) & (df['Uwind [m/s]'].notnull()) & 
    #                 (df['Vwind [m/s]'].notnull()) & (df['Temperature [C]'].notnull()) & (df['Dewpoint [C]'].notnull()) & (df['RH [%]'] >= 0)].copy()
    #         #^^^ using .copy() to prevent chain-indexing --> https://www.dataquest.io/blog/settingwithcopywarning/

    if len(df_use) != 0:

        #create one single height column, prioritizing GPS altitude over geopotential altitude
        heights_use = []
        for i in range(len(df_use)):
            gps_height = df_use['GPS Altitude [m]'].iloc[i]
            if gps_height > 0:  #i.e., if the GPS height value is not NaN, use GPS height
                heights_use.append(gps_height)
            else:
                heights_use.append(df_use['Geopotential Altitude [m]'].iloc[i])  #if the GPS height value is NaN, use the geopotential height
        df_use['Heights Use'] = heights_use  #needed for proper rounding of height Series....for some reason
        
        df_use['Potential Temperature [K]'] = mpcalc.potential_temperature(df_use['Pressure [mb]'].values * units('hPa'), df_use['Temperature [C]'].values * units('celsius')).m

        #convert the good, relevant radiosonde data to a dataframe and add to the final radiosonde CSV file           
        radio_clean_df = pd.DataFrame(columns = ['Time [UTC]', 'Height [m]', 'Pressure [mb]', 'U Comp of Wind [m/s]', 'V Comp of Wind [m/s]',
                                                'Wind Speed [m/s]', 'Wind Direction [deg]', 'Temperature [C]', 'Dew Point [C]',
                                                'Potential Temperature [K]', 'Relative Humidity [%]', 'Latitude [deg]', 'Longitude [deg]'])
        radio_clean_df['Time [UTC]'] = [radio_full_time] * len(df_use)  #a list of len(df_use) with the same radio_full_time value
        radio_clean_df['Height [m]'] = np.round(list(df_use['Heights Use'])[::-1], 2)
        radio_clean_df['Pressure [mb]'] = np.round(list(df_use['Pressure [mb]'])[::-1], 2)
        radio_clean_df['U Comp of Wind [m/s]'] = np.round(list(df_use['Uwind [m/s]'])[::-1], 2)
        radio_clean_df['V Comp of Wind [m/s]'] = np.round(list(df_use['Vwind [m/s]'])[::-1], 2)
        radio_clean_df['Wind Speed [m/s]'] = np.round(list(df_use['Wspd [m/s]'])[::-1], 2)
        radio_clean_df['Wind Direction [deg]'] = np.round(list(df_use['Wdir [deg]'])[::-1], 2)
        radio_clean_df['Temperature [C]'] = np.round(list(df_use['Temperature [C]'])[::-1], 2)
        radio_clean_df['Dew Point [C]'] = np.round(list(df_use['Dewpoint [C]'])[::-1], 2)
        radio_clean_df['Potential Temperature [K]'] = np.round(list(df_use['Potential Temperature [K]'])[::-1], 2)
        radio_clean_df['Relative Humidity [%]'] = np.round(list(df_use['RH [%]'])[::-1], 2)
        radio_clean_df['Latitude [deg]'] = np.round(list(df_use['Latitude [deg]'])[::-1], 7)
        radio_clean_df['Longitude [deg]'] = np.round(list(df_use['Longitude [deg]'])[::-1], 7)

        if first_file:
            radio_clean_df.to_csv(radio_final_name, index = False)
            first_file = False
        else:
            df_all = pd.read_csv(radio_final_name)
            df_total = pd.concat([df_all, radio_clean_df], ignore_index = True)  #concatenates fields with same heading
            df_total.to_csv(radio_final_name, index = False)
            

In [ ]:
#plot up the available, good radiosonde data for the given time range
df_all = pd.read_csv(radio_final_name)

df_winds = df_all[(df_all['U Comp of Wind [m/s]'].notnull()) & (df_all['V Comp of Wind [m/s]'].notnull())].copy()
df_moisture = df_all[(df_all['Temperature [C]'].notnull()) & (df_all['Dew Point [C]'].notnull()) & (df_all['Relative Humidity [%]'].notnull())].copy()

radio_fig, radio_axs = plt.subplots(nrows=1, ncols=2, figsize = (35,20))
radio_fig.subplots_adjust(wspace=0.2)

radio_moisture_x_ax = pd.to_datetime(df_moisture['Time [UTC]'])
radio_moisture_y_ax = df_moisture['Height [m]']
radio_axs[0].scatter(radio_moisture_x_ax, radio_moisture_y_ax, s=15, c='k')
radio_axs[0].set_ylim([0, 8500])
radio_axs[0].set_yticks(np.arange(0, 8501, 500))
radio_axs[0].tick_params(axis='x', rotation = 50)
radio_axs[0].tick_params(labelsize=18)
radio_axs[0].grid(True)
radio_axs[0].set_xlabel('Time [UTC]', fontsize=30)
radio_axs[0].set_ylabel('Height [m]', fontsize=30)
radio_axs[0].set_title('Radiosonde Moisture Availability', fontsize=40)
radio_axs[0].xaxis.set_major_formatter(mpl.dates.DateFormatter("%H:%M"))
#radio_axs[0].gcf().set_size_inches(10,13)

radio_winds_x_ax = pd.to_datetime(df_winds['Time [UTC]'])
radio_winds_y_ax = df_winds['Height [m]']
radio_axs[1].scatter(radio_winds_x_ax, radio_winds_y_ax, s=15, c='k')
radio_axs[1].set_ylim([0, 8500])
radio_axs[1].set_yticks(np.arange(0, 8501, 500))
radio_axs[1].tick_params(axis='x', rotation = 50)
radio_axs[1].tick_params(labelsize=18)
radio_axs[1].grid(True)
radio_axs[1].set_xlabel('Time [UTC]', fontsize=30)
radio_axs[1].set_ylabel('Height [m]', fontsize=30)
radio_axs[1].set_title('Radiosonde Wind Availability', fontsize=40)
radio_axs[1].xaxis.set_major_formatter(mpl.dates.DateFormatter("%H:%M"))
#radio_axs[1].gcf().set_size_inches(10,13)

# #plot up same figure but with wind barbs instead of dots
# radio_u = df_all['U Comp of Wind [m/s]']
# radio_v = df_all['V Comp of Wind [m/s]']
# radio_axs[1].barbs(radio_x_ax, radio_y_ax, radio_u, radio_v, fill_empty = True, pivot='middle', sizes=dict(emptybarb=0.075), barbcolor = 'b')
# #add "np.sqrt(radio_u**2 + radio_v**2)" to above line to color code barbs by speed
# radio_axs[1].tick_params(axis='x', rotation = 50)
# radio_axs[1].tick_params(labelsize=18)
# radio_axs[1].grid(True)
# radio_axs[1].set_xlabel('Time [UTC]', fontsize=30)
# radio_axs[1].set_ylabel('Height [m]', fontsize=30)
# radio_axs[1].set_title('Radiosonde 2-D Wind at Given Times and Heights', fontsize=40)
# radio_axs[1].xaxis.set_major_formatter(mpl.dates.DateFormatter("%H:%M"))
# #radio_axs[1].gcf().set_size_inches(20,25)

radio_name = os.path.join(day_folder, "Radiosonde_avail_moisture_and_winds_" + file_date + ".png")
plt.savefig(radio_name, bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
plt.close()


In [ ]:
#make theta, theta-e, and virtual theta plots for each radiosonde to figure out which ones should be omitted
#exclude radiosonde profiles with large vertical data gaps and/or frequent, graphically visible anomalous spikes
    
radio_csv = pd.read_csv(radio_final_name)
radio_times = sorted(radio_csv['Time [UTC]'].unique())  #sorted() = goes through files in alphabetical order

line_types = ['b-', 'r-', 'k-', 'y-', 'b--', 'r--', 'k--', 'y--', 'c-', 'c--', 'm-', 'm--', 'b:', 'r:', 'k:', 'y:', 'c:', 'm:', 'b-.', 'r-.', 'k-.', 'y-.', 'c-.', 'm-.', 'g-', 'g--', 'g:', 'g-.',
              'b-', 'r-', 'k-', 'y-', 'b--', 'r--', 'k--', 'y--', 'c-', 'c--', 'm-', 'm--', 'b:', 'r:', 'k:', 'y:', 'c:', 'm:', 'b-.', 'r-.', 'k-.', 'y-.', 'c-.', 'm-.', 'g-', 'g--', 'g:', 'g-.']

#Potential Temperature 
fig = plt.figure(figsize=(15,15))   
    
line_index = 0
for time in radio_times:
    rel_data = radio_csv[radio_csv['Time [UTC]'] == time].copy()

    pres = rel_data['Pressure [mb]']
    hght = rel_data['Height [m]']
    tmpc = rel_data['Temperature [C]']
    dwpc = rel_data['Dew Point [C]']
    wspd = 1.94384449 * rel_data['Wind Speed [m/s]']  #converts m/s to knots (also in SHARPpy sharptab.utils script)
    wdir = rel_data['Wind Direction [deg]']

    plt.plot(rel_data['Potential Temperature [K]'], rel_data['Pressure [mb]'], line_types[line_index], label = time[11:])
    plt.xlabel("Potential Temperature [K]", fontsize = 25)
    plt.ylabel("Pressure [mb]", fontsize = 25)
    plt.ylim([1050,190])  #inverts y-axis (pressure)
    plt.yticks(np.arange(1000,190,-50))
    plt.xlim([290, 380])
    plt.xticks(np.arange(290,380.1,10))
    plt.tick_params(labelsize = 15)
    plt.legend(fontsize = 'xx-large')
    plt.grid(True)
    plt.title(file_date + ' Radiosonde Theta Profiles', fontsize = 30)
    plt.savefig(os.path.join(day_folder, 'theta_profiles_radio.png'), bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
    line_index = line_index + 1
plt.close()
  

#Equivalent Potential Temperature
fig = plt.figure(figsize=(15,15))

line_index = 0
for time in radio_times:
    rel_data = radio_csv[radio_csv['Time [UTC]'] == time].copy()
    rel_data2 = rel_data.iloc[::-1]  #reverses the dataframe (row-based) to go from surface to upper-level

    pres = rel_data2['Pressure [mb]']
    hght = rel_data2['Height [m]']
    tmpc = rel_data2['Temperature [C]']
    dwpc = rel_data2['Dew Point [C]']
    wspd = 1.94384449 * rel_data2['Wind Speed [m/s]']  #converts m/s to knots (also in SHARPpy sharptab.utils script)
    wdir = rel_data2['Wind Direction [deg]']

    try:
        prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=True)
    except:
        prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=False)
            
    plt.plot(prof.thetae.data, prof.pres.data, line_types[line_index], label = time[11:])
    plt.xlabel("Equivalent Potential Temperature [K]", fontsize = 25)
    plt.ylabel("Pressure [mb]", fontsize = 25)
    plt.ylim([1050,190])  #inverts y-axis (pressure)
    plt.yticks(np.arange(1000,190,-50))
    plt.xlim([320, 380])
    plt.xticks(np.arange(320,380.1,5))
    plt.tick_params(labelsize = 15)
    plt.legend(fontsize = 'xx-large')
    plt.grid(True)
    plt.title(file_date + ' Radiosonde Theta-E Profiles', fontsize = 30)
    plt.savefig(os.path.join(day_folder, 'thetaE_profiles_radio.png'), bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
    line_index = line_index + 1
plt.close()
    
    
#Virtual Potential Temperature    
fig = plt.figure(figsize=(15,15))
    
line_index = 0
for time in radio_times:
    rel_data = radio_csv[radio_csv['Time [UTC]'] == time].copy()
    rel_data2 = rel_data.iloc[::-1]  #reverses the dataframe (row-based) to go from surface to upper-level

    pres = rel_data2['Pressure [mb]']
    hght = rel_data2['Height [m]']
    tmpc = rel_data2['Temperature [C]']
    dwpc = rel_data2['Dew Point [C]']
    wspd = 1.94384449 * rel_data2['Wind Speed [m/s]']  #converts m/s to knots (also in SHARPpy sharptab.utils script)
    wdir = rel_data2['Wind Direction [deg]']

    try:
        prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=True)
    except:
        prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=False)

    thetav = thermo.theta(prof.pres.data, thermo.virtemp(prof.pres.data, prof.tmpc.data, prof.dwpc.data))   
    thetav = thermo.ctok(thetav)  #convert from Celsius to Kelvin
    plt.plot(thetav, prof.pres.data, line_types[line_index], label = time[11:])
    plt.xlabel("Virtual Potential Temperature [K]", fontsize = 25)
    plt.ylabel("Pressure [mb]", fontsize = 25)
    plt.ylim([1050,190])  #inverts y-axis (pressure)
    plt.yticks(np.arange(1000,190,-50))
    plt.xlim([290, 380])
    plt.xticks(np.arange(290,380.1,10))
    plt.tick_params(labelsize = 15)
    plt.legend(fontsize = 'xx-large')
    plt.grid(True)
    plt.title(file_date + ' Radiosonde Theta-V Profiles', fontsize = 30)
    plt.savefig(os.path.join(day_folder, 'thetaV_profiles_radio.png'), bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
    line_index = line_index + 1
plt.close()


In [ ]:
#plot radiosonde skew-T using MetPy(don't need to change anything in this cell)
def plot_skewTs(file_date, plot_hodograph = True):
    """ Make skew-T/hodograph figures for a given day's radiosonde profiles
    
    PARAMETERS
    ----------
    file_date : the day (YYYYMMDD, string format) for which you want to plot radiosonde profile skew-Ts
    plot_hodograph : determines whether or not to plot an inset hodograph (True/False)
    
    RETURNS
    ----------
    fig : matplotlib skew-T figures for each radiosonde in the given day's "final_radiosonde_YYYYMMDD.csv" file
    
    """
    
    day_folder = os.path.join(os.getcwd(), file_date)   
    radiosonde_folder = os.path.join(day_folder, 'Radiosonde_files')
    radio_final_name = os.path.join(day_folder, 'final_radiosonde_' + file_date + '.csv')
    
    radio_csv = pd.read_csv(radio_final_name)
    radio_times = radio_csv['Time [UTC]'].unique()  #sorted() = goes through files in alphabetical order
    
    #initialize some plot visualizations
    mpl.rcParams['font.family'] = 'arial'
    mpl.rcParams['font.size'] = 15
    mpl.rcParams['ytick.labelsize'] = 14
    mpl.rcParams['xtick.labelsize'] = 14

    for time in radio_times:
        #print (time)    #for debugging purposes
        
        df0 = radio_csv[radio_csv['Time [UTC]'] == time].copy()
        df0 = df0[df0['Pressure [mb]'] >= 100].copy()   #added for radisondes (not dropsondes) to help avoid "UserWarning: Duplicate pressure(s)" warning
        df = df0.iloc[::-1]  #reverses the dataframe (row-based) to go from surface to upper-level

        #add appropriate units to the pressure, temperature, dewpoint, and wind data
        pres = df['Pressure [mb]'].values * units.hPa   #hPa = mb
        #filtered_pres = pres[::2]      #only every other pressure value for quicker profile/skew-T creation
        temp = df['Temperature [C]'].values * units.degC
        #filtered_temp = temp[::2]      ##only every other temp. value for quicker skew-T creation
        dwpt = df['Dew Point [C]'].values * units.degC
        #wnd_spd = df['Wind Speed [m/s]'].values * units('m/s')
        wnd_spd = (df['Wind Speed [m/s]'].values * units('m/s')).to(units.knots)  #convert wind speed to knots
        wnd_dir = df['Wind Direction [deg]'].values * units.deg
        #hght = df['Height [m]'].values * units.meter

        #calculate the parcel path/profile at the near-surface for the given environment
        
        #profile = mpcalc.parcel_profile(pres, temp[0], dwpt[0])  #returns temps in Kelvin
        try:
            profile = mpcalc.parcel_profile(pres, temp[0], dwpt[0])  #returns temps in Kelvin
        except:
            print (f'Could not plot {time} skew-T, likely because this radiosonde contains no moisture data or pressure increases between at least two points in the sounding. Using scipy.signal.medfilt may fix the latter.')
            continue
            
        profile = profile.to('degC')

        #calculate the LCL and wind components
        lcl = mpcalc.lcl(pres[0], temp[0], dwpt[0])
        wind_comps = mpcalc.wind_components(wnd_spd, wnd_dir)  #returns u,v values in whatever unit wnd_spd is in
        u = wind_comps[0]
        v = wind_comps[1]

        #initialize the figure
        fig = plt.figure(figsize = (12,12))       

        #Initialize the skew-T figure/subplot
        skew = mplots.SkewT(fig)

        #Plot the data for the skew-T
        skew.plot(pres, temp, 'darkorange', linewidth = 2)
        skew.plot(pres, dwpt, 'cornflowerblue', linewidth = 2)
        #skew.plot(lcl[0], lcl[1], 'yellow', marker = '*', markeredgecolor = 'k', markersize = 14)  #plot the LCL as a yellow star
        #skew.plot(filtered_pres, profile, 'k', linewidth = 2)
        skew.plot(pres, profile, 'k', linewidth = 2)
        skew.plot_barbs(pres[::60], u[::60], v[::60])
        #skew.shade_cape(filtered_pres, filtered_temp, profile)
        #skew.shade_cin(filtered_pres, filtered_temp, profile, dwpt[::2])
        skew.shade_cape(pres, temp, profile, alpha = 0.2)
        skew.shade_cin(pres, temp, profile, alpha = 0.2)
        skew.plot_dry_adiabats(t0 = np.arange(-90, 321, 10) * units.degC, alpha = 0.3)   #range is large to cover whole plot for all possible profiles
        skew.plot_moist_adiabats(t0 = np.arange(-90, 81, 10) * units.degC, alpha = 0.3)  #range is large to cover whole plot for all possible profiles
        skew.ax.set_xlim(-50,40)
        skew.ax.set_ylim(1000,200)
        skew.ax.set_title(f'{time} Radiosonde Skew-T Diagram and Hodograph') #title created based on radiosonde time
        skew.ax.set_xlabel('T [$\\degree$C]')
        skew.ax.set_ylabel('Pressure [hPa]')
        
        #if just one sounding text file is inputted, then plot a hodograph in the upper right-hand corner of the figure
        if plot_hodograph:
            axh = inset_axes(skew.ax, '35%', '35%', loc = 'upper right')
            h = mplots.Hodograph(axh, component_range = 80.)
            h.add_grid(increment = 20)
            
            try:
                h.plot_colormapped(u, v, wnd_spd);  # Plot a line colored by wind speed
            except:
                fig.text(0.755, 0.643, 'No Wind Data', horizontalalignment='center', 
                         verticalalignment='center', fontsize = 20)
        
        save_name = os.path.join(day_folder, 'skewt_' + time[11:13] + time[14:16] + time[17:19] + '_radio.png')
        #plt.show()
        plt.savefig(save_name, bbox_inches = 'tight')  #bbox_inches = 'tight' will clip any additional white space around the image
        plt.close()
        
        #decrease file size of the image by 4x without noticeable image effects (if using Matplotlib)!
        #(good to use if you're producing a lot of images, see https://www.youtube.com/watch?v=fzhAseXp5B4)
        im = Image.open(save_name)
        try:
            im2 = im.convert('P', palette = Image.Palette.ADAPTIVE)
        except:
            im2 = im.convert('P')  #use this for older version of PIL/Pillow if the above line doesn't work, though this line will have isolated, extremely minor image effects due to only using 256 colors instead of the 3-part RGB scale
        im2.save(save_name)
        im.close()
        im2.close()

plot_skewTs(file_date)


In [ ]:
print ('Done!')